# Notebook 03 — Entrenamiento y Comparación de Modelos

**Proyecto:** IntApp — Predicción del riesgo de lesión deportiva  
**Autor:** Roberto Franco  
**Director:** Javier Sánchez  
**Universidad:** Universidad Europea de Madrid  

---


## 1. Introducción

En este notebook entrenamos y comparamos **tres algoritmos de Machine Learning** para predecir el nivel de riesgo de lesión de miembro inferior en deportistas:

- **Random Forest**: conjunto de árboles de decisión, robusto ante outliers.
- **Regresión Logística**: modelo lineal, interpretable y rápido.
- **Gradient Boosting**: ensamblado secuencial, habitualmente el más preciso.

La métrica principal es el **F1-score macro** sobre las tres clases (bajo / medio / alto). Se aplica además una **estrategia clínica asimétrica** para minimizar los falsos negativos en la clase 'alto': en prevención de lesiones, no detectar un deportista de alto riesgo es mucho más grave que sobre-alertar a uno sano.

> **Nota:** la categoría `no_concluyente` (NRS > 5) se gestiona como regla pre-modelo: si el deportista tiene dolor agudo activo, la app muestra el aviso directamente sin pasar por el clasificador.


## 2. Configuración e importaciones


In [ ]:
import sys
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
warnings.filterwarnings('ignore')

# Añadir la raíz del proyecto al path
RAIZ_PROYECTO = Path(os.getcwd()).parent
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

from src.generador_datos import generar_dataset
from src.preprocesador import preprocesar
from src.modelo import (
    dividir_datos,
    evaluar_modelo,
    comparar_modelos,
    guardar_modelo,
    entrenar_gradient_boosting,
    calibrar_umbral_alto,
    validar_cruzada,
    CalibradorUmbralAlto,
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score

print(f'Raíz del proyecto: {RAIZ_PROYECTO}')
print('Importaciones correctas.')


## 3. Generación y preprocesamiento de datos

Generamos el dataset combinando **3 semillas × 700 muestras = 2.100 deportistas**. Usar varias semillas aumenta la variabilidad y mejora la generalización del modelo.


In [ ]:
print('Generando dataset combinado (3 semillas x 700 muestras)...')
dfs = []
for seed in [42, 123, 2025]:
    df_seed = generar_dataset(n_deportistas=700, semilla=seed)
    dfs.append(df_seed)
df_crudo = pd.concat(dfs, ignore_index=True)

# 'no_concluyente' es una regla clínica pre-modelo (NRS > 5).
# Se filtra aquí: la app la gestiona antes de llamar al clasificador.
df_crudo = df_crudo[df_crudo['riesgo_lesion'] != 'no_concluyente'].copy()
df_crudo = df_crudo.reset_index(drop=True)

print(f'Dataset: {df_crudo.shape[0]} filas x {df_crudo.shape[1]} columnas')
print('\nDistribución de clases:')
print(df_crudo['riesgo_lesion'].value_counts())
df_crudo.head(3)


In [ ]:
# Pipeline de preprocesamiento:
# 1. Ratios clínicos (H:Q, ADD/ABD, fuerza media normalizada)
# 2. Asimetrias bilaterales (%)
# 3. Encoding de categoricas (genero, nivel_actividad, perfil_exigencia)
# 4. Eliminacion de columnas auxiliares (score_total, confianza, reglas)
# 5. Normalizacion StandardScaler
df_procesado, scaler = preprocesar(df_crudo)

print(f'Dataset preprocesado: {df_procesado.shape[0]} filas x {df_procesado.shape[1]} columnas')
print(f'Features para el modelo: {df_procesado.shape[1] - 1}')
print(f'NaN en el dataset: {df_procesado.isnull().sum().sum()}')

# Guardar datos procesados
DIR_PROCESADOS = RAIZ_PROYECTO / 'datos' / 'procesados'
DIR_PROCESADOS.mkdir(parents=True, exist_ok=True)
df_procesado.to_csv(DIR_PROCESADOS / 'dataset_procesado.csv', index=False)
print(f'Dataset procesado guardado.')


## 4. División en entrenamiento y test

- **Entrenamiento (80%):** datos que el modelo usa para aprender.
- **Test (20%):** datos que el modelo **nunca ha visto**, usados para las métricas finales.

Se usa `stratify` para preservar la proporción de clases en ambos conjuntos.


In [ ]:
X_train, X_test, y_train, y_test = dividir_datos(
    df_procesado,
    columna_objetivo='riesgo_lesion',
    test_size=0.20,
    semilla=42,
)

print(f'Train: {len(X_train)} muestras | Test: {len(X_test)} muestras | Features: {X_train.shape[1]}')
print()
dist_train = y_train.value_counts(normalize=True).mul(100).round(1)
dist_test  = y_test.value_counts(normalize=True).mul(100).round(1)
pd.DataFrame({'Train (%)': dist_train, 'Test (%)': dist_test}).reindex(['bajo','medio','alto'])


## 5. Entrenamiento de los modelos

Se usan hiperparámetros optimizados:
- **Random Forest:** 500 árboles, profundidad 12, pesos balanceados.
- **Regresión Logística:** regularización C=0.5, pesos balanceados.
- **Gradient Boosting:** 400 estimadores, learning rate 0.05, profundidad 4, subsample 0.8.


In [ ]:
print('=' * 55)
print('Entrenando 1/3: Random Forest')
print('=' * 55)
t0 = time.time()
modelo_rf = RandomForestClassifier(
    n_estimators=500, max_depth=12, min_samples_leaf=2,
    max_features='sqrt', class_weight='balanced',
    random_state=42, n_jobs=-1,
)
modelo_rf.fit(X_train, y_train)
tiempo_rf = time.time() - t0
print(f'Tiempo: {tiempo_rf:.2f}s')


In [ ]:
print('=' * 55)
print('Entrenando 2/3: Regresion Logistica')
print('=' * 55)
t0 = time.time()
modelo_rl = LogisticRegression(
    C=0.5, class_weight='balanced',
    max_iter=2000, solver='lbfgs', random_state=42,
)
modelo_rl.fit(X_train, y_train)
tiempo_rl = time.time() - t0
print(f'Tiempo: {tiempo_rl:.2f}s')


In [ ]:
print('=' * 55)
print('Entrenando 3/3: Gradient Boosting')
print('=' * 55)
t0 = time.time()
pesos_balanced = compute_sample_weight('balanced', y_train)
modelo_gb = GradientBoostingClassifier(
    n_estimators=400, learning_rate=0.05, max_depth=4,
    subsample=0.8, min_samples_leaf=2, random_state=42,
)
modelo_gb.fit(X_train, y_train, sample_weight=pesos_balanced)
tiempo_gb = time.time() - t0
print(f'Tiempo: {tiempo_gb:.2f}s')
print()
pd.DataFrame({
    'Modelo': ['Random Forest', 'Regresion Logistica', 'Gradient Boosting'],
    'Tiempo (s)': [round(tiempo_rf, 2), round(tiempo_rl, 2), round(tiempo_gb, 2)],
})


## 6. Evaluacion de los modelos

Evaluamos cada modelo sobre el conjunto de test. La metrica principal es el **F1-score macro** (promedio no ponderado de las 3 clases).


### 6.1 Random Forest


In [ ]:
resultado_rf = evaluar_modelo(modelo_rf, X_test, y_test, nombre='Random Forest')


### 6.2 Regresion Logistica


In [ ]:
resultado_rl = evaluar_modelo(modelo_rl, X_test, y_test, nombre='Regresion Logistica')


### 6.3 Gradient Boosting


In [ ]:
resultado_gb = evaluar_modelo(modelo_gb, X_test, y_test, nombre='Gradient Boosting')


## 7. Comparacion entre modelos

Tabla resumen y grafico de barras con el F1-score macro de cada modelo.


In [ ]:
CLASES = ['bajo', 'medio', 'alto']

resultados_todos = {
    'Random Forest':       resultado_rf,
    'Regresion Logistica': resultado_rl,
    'Gradient Boosting':   resultado_gb,
}

# comparar_modelos devuelve un DataFrame con las metricas y genera el grafico
df_comparacion = comparar_modelos(resultados_todos)

# Identificar el mejor modelo
mejor_nombre = df_comparacion['F1 macro'].idxmax()
mapa_modelos = {
    'Random Forest':       modelo_rf,
    'Regresion Logistica': modelo_rl,
    'Gradient Boosting':   modelo_gb,
}
mejor_modelo_estandar = mapa_modelos[mejor_nombre]
print(f'Mejor modelo por F1-score macro: {mejor_nombre}')


## 7b. Validación cruzada (5-fold estratificada)

Antes de la estrategia clínica asimétrica, ejecutamos una **validación cruzada de 5 folds** sobre el conjunto de entrenamiento.  
Esto estima la varianza del rendimiento más allá del split 80/20 único y detecta posible sobreajuste.


In [ ]:
df_cv = validar_cruzada(X_train, y_train, semilla=42, n_folds=5)
df_cv


## 8. Estrategia clínica asimétrica

El modelo estándar clasifica incorrectamente como 'bajo' o 'medio' a algunos deportistas de alto riesgo.  
En prevención de lesiones, **este es el error más peligroso** (falso negativo).

Se aplica una estrategia en dos pasos:

1. **Pesos de clase en el entrenamiento** — `entrenar_gradient_boosting(..., peso_extra_alto=2.5)` penaliza los errores en 'alto' de forma automática y reproducible.
2. **Umbral de decisión calibrado** — `calibrar_umbral_alto()` encuentra el umbral que maximiza el recall de 'alto' sujeto a **dos restricciones simultáneas**: F1 macro ≥ 0.58 **y** precisión 'alto' ≥ 0.60.

> *Justificación: el coste de un falso negativo (no detectar un deportista en riesgo) puede ser una lesión grave.  
> La restricción de precisión evita umbrales agresivos (p.ej. 0.10) que generarían una tasa de falsas alarmas clínicamente inaceptable.*


In [ ]:
# Val split interno para calibrar el umbral (nunca toca X_test)
X_tr_gb, X_val_gb, y_tr_gb, y_val_gb = train_test_split(
    X_train, y_train, test_size=0.20, random_state=42, stratify=y_train
)
print(f'Split calibración — Entrenamiento: {len(X_tr_gb)} | Validación: {len(X_val_gb)}')

# Entrenar GB clínico con peso extra para 'alto'
print('\nEntrenando GB clínico (peso_extra_alto=2.5)...')
t0 = time.time()
modelo_gb_cal_base = entrenar_gradient_boosting(X_tr_gb, y_tr_gb, semilla=42, peso_extra_alto=2.5)
print(f'Entrenado en {time.time()-t0:.2f}s')

# Calibrar umbral sobre val set con restricciones clínicas
umbral_optimo = calibrar_umbral_alto(
    modelo_gb_cal_base, X_val_gb, y_val_gb,
    min_f1_macro=0.58, min_precision_alto=0.60,
)
print(f'Umbral óptimo seleccionado: {umbral_optimo:.2f}')


In [ ]:
# Wrapper sklearn con umbral calibrado
modelo_gb_cal = CalibradorUmbralAlto(modelo_gb_cal_base, umbral_alto=umbral_optimo)

# Comparativa: GB estándar vs GB calibrado
CLASES = ['bajo', 'medio', 'alto']
print('=' * 62)
print('  GB ESTÁNDAR  vs  GB CALIBRADO (umbral calibrado con restricciones)')
print('=' * 62)
print('\n--- GB estándar ---')
print(classification_report(
    y_test, modelo_gb.predict(X_test),
    labels=CLASES, target_names=CLASES, zero_division=0
))
print(f'--- GB Calibrado (umbral alto = {umbral_optimo:.2f}) ---')
print(classification_report(
    y_test, modelo_gb_cal.predict(X_test),
    labels=CLASES, target_names=CLASES, zero_division=0
))

# Métrica clave: falsos negativos alto → bajo
y_pred_gb_std = modelo_gb.predict(X_test)
y_pred_cal    = modelo_gb_cal.predict(X_test)
fn_estandar  = sum(1 for r, p in zip(y_test, y_pred_gb_std) if r == 'alto' and p == 'bajo')
fn_calibrado = sum(1 for r, p in zip(y_test, y_pred_cal)    if r == 'alto' and p == 'bajo')
total_alto   = sum(y_test == 'alto')

print('=' * 62)
print('  MÉTRICA CLÍNICA CLAVE: falsos negativos alto → bajo')
print('=' * 62)
print(f'  GB estándar  : {fn_estandar} de {total_alto} casos ({fn_estandar/total_alto*100:.1f}%)')
print(f'  GB Calibrado : {fn_calibrado} de {total_alto} casos ({fn_calibrado/total_alto*100:.1f}%)')
print(f'\n  Reducción de falsos negativos: {fn_estandar - fn_calibrado} casos menos sin detectar')


### 8.1 Intervalo de confianza bootstrap (recall_alto)

El test set tiene ≈ 400 muestras con ~80 de clase 'alto'. Un solo recall puntual no informa de su incertidumbre. El bootstrap remuestrea 1 000 veces con reemplazamiento y devuelve el IC 95% por percentiles, sin asumir normalidad.

In [ ]:
from src.modelo import bootstrap_ic

_recall_fn = lambda yt, yp: recall_score(yt, yp, labels=['alto'], average='macro', zero_division=0)
ic_lo, ic_hi = bootstrap_ic(
    y_test, modelo_gb_cal.predict(X_test), _recall_fn,
    n_iter=1000, nivel_confianza=0.95, semilla=42,
)
print(f'Recall alto (test)  : {_recall_fn(y_test, modelo_gb_cal.predict(X_test)):.4f}')
print(f'IC 95% bootstrap    : [{ic_lo:.4f}, {ic_hi:.4f}]')
print(f'Amplitud del IC     : {ic_hi - ic_lo:.4f}  '
      f'(amplitud alta → test set pequeño, interpretar con cautela)')

## 9. Guardado del modelo final

Se guarda **`CalibradorUmbralAlto`** como modelo definitivo: un wrapper que encapsula el GB clínico junto con el umbral calibrado. Es el que se carga en la app Streamlit para evaluar nuevos deportistas — `predict()` aplica automáticamente el umbral óptimo.

In [ ]:
DIR_MODELOS = RAIZ_PROYECTO / 'modelos'
DIR_MODELOS.mkdir(parents=True, exist_ok=True)

# Modelo principal: CalibradorUmbralAlto (wrapper GB clínico + umbral calibrado)
guardar_modelo(modelo_gb_cal, DIR_MODELOS / 'mejor_modelo.pkl')
print('Guardado: mejor_modelo.pkl  →  CalibradorUmbralAlto (GB clínico, umbral calibrado)')

# Modelos estándar para comparación y análisis SHAP
joblib.dump(modelo_rf, DIR_MODELOS / 'modelo_rf.joblib')
joblib.dump(modelo_rl, DIR_MODELOS / 'modelo_rl.joblib')
joblib.dump(modelo_gb, DIR_MODELOS / 'modelo_gb.joblib')
print('Guardados: modelo_rf.joblib, modelo_rl.joblib, modelo_gb.joblib')

# Scaler necesario para preprocesar nuevos deportistas en la app
joblib.dump(scaler, DIR_MODELOS / 'scaler.pkl')
print('Guardado: scaler.pkl')

# Resumen final
y_pred_cal_test = modelo_gb_cal.predict(X_test)
f1_cal    = f1_score(y_test, y_pred_cal_test, average='macro', labels=CLASES, zero_division=0)
rec_alto  = recall_score(y_test, y_pred_cal_test, labels=['alto'], average='macro', zero_division=0)
prec_alto = precision_score(y_test, y_pred_cal_test, labels=['alto'], average='macro', zero_division=0)
fn_cal    = sum(1 for r, p in zip(y_test, y_pred_cal_test) if r == 'alto' and p == 'bajo')

print()
print('=== RESUMEN FINAL ===')
print(f'Modelo seleccionado    : CalibradorUmbralAlto (GB clínico)')
print(f'Umbral clase alto      : {umbral_optimo:.2f}  (precision≥0.60, F1≥0.58)')
print(f'F1-macro test          : {f1_cal:.4f}')
print(f'Recall clase alto      : {rec_alto:.4f}')
print(f'Precisión clase alto   : {prec_alto:.4f}')
print(f'Falsos neg. alto→bajo  : {fn_cal} de {total_alto} ({fn_cal/total_alto*100:.1f}%)')
print(f'CV 5-fold GB (F1)      : {df_cv.loc["Gradient Boosting","F1_media"]:.4f} ± {df_cv.loc["Gradient Boosting","F1_std"]:.4f}')
print(f'CV 5-fold GB (Rec.alto): {df_cv.loc["Gradient Boosting","Recall_alto_media"]:.4f} ± {df_cv.loc["Gradient Boosting","Recall_alto_std"]:.4f}')
print(f'Dataset                : 3 semillas x 700 = 2.100 muestras')
